In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return start

repo_root = _find_repo_root(Path.cwd())
workspace_root = repo_root.parent
candidate_src_dirs = [
    repo_root / "src",
    workspace_root / "abstractgraph" / "src",
    workspace_root / "abstractgraph-ml" / "src",
    workspace_root / "abstractgraph-generative" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.exists():
        src_str = str(src_dir)
        if src_str not in sys.path:
            sys.path.insert(0, src_str)


# Interpolate Graph Demo
Load PubChem assay graphs and render a small sample.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [ ]:
def interactive_virtual_walk(
    seed,
    generator,
    feasibility_estimator,
    display_graphs,
    *,
    n_virtual_candidates: int = 30,
    max_virtual_cut_evaluations: int = 200,
    max_attempts: int = 5,
):
    """Interactively explore virtual-cut candidates from a seed graph.

    Args:
        seed: Starting NetworkX graph.
        generator: AutoregressiveGraphGenerator instance.
        feasibility_estimator: Feasibility estimator used to filter candidates.
        display_graphs: Callable to visualize a list of graphs.
        n_virtual_candidates: Number of virtual candidates to propose each step.
        max_virtual_cut_evaluations: Max virtual-cut evaluations per step.
        max_attempts: Number of attempts to find candidates before stopping.

    Returns:
        The last selected seed graph.
    """
    step = 0
    while True:
        virtual_samples = []
        for _attempt in range(max(1, int(max_attempts))):
            virtual_samples = generator.propose_virtual_candidates(
                seed,
                n_virtual_candidates=n_virtual_candidates,
                max_virtual_cut_evaluations=max_virtual_cut_evaluations,
            )
            virtual_samples = feasibility_estimator.filter(virtual_samples)
            if virtual_samples:
                break
        if not virtual_samples:
            print("No virtual candidates available; stopping.")
            _ = display_graphs([seed])
            return seed
        print(f"Step {step}: {len(virtual_samples)} candidates")
        displayed = [seed] + virtual_samples
        _ = display_graphs(displayed)
        choice = input("Pick displayed index (0 = seed, 1..N) or Q to quit: ").strip()
        if choice.lower() == "q":
            _ = display_graphs([seed])
            return seed
        try:
            idx = int(choice)
        except ValueError:
            print("Invalid input; please enter a number or Q.")
            continue
        if idx < 0 or idx >= len(displayed):
            print("Index out of range; try again.")
            continue
        if idx == 0:
            step += 1
            continue
        seed = displayed[idx]
        step += 1

# Example usage
# seed = generator.donors[0]
# seed = interactive_virtual_walk(seed, generator, feasibility_estimator, display_graphs)


In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
import networkx as nx
from IPython.display import display, clear_output
import itertools
import math
import hashlib
import threading
from abstractgraph_generative.rewrite import virtual_cut_signature
from abstractgraph.hashing import GraphHashDeduper
from coco_grape.visualizer.mol_display import draw_molecules

_NODE_PALETTE = [
    "#70ADBF",
    "#FFB700",
    "#2D37FF",
    "#FF2A2A",
    "#A16207",
    "#4ACA73",
    "#2FFF00",
    '#E7C6FF',
    '#BDE0FE',
    '#FFD6A5',
]

_EDGE_PALETTE = [
    "#90BBE7",
    "#9BACB1",
    "#31AAB3",
    "#3C3F83",
    "#7C3AED",
    "#A16207",
]
_NODE_COLOR_MAP = {}
_EDGE_COLOR_MAP = {}
_NODE_LABEL_ORDER = None
_EDGE_LABEL_ORDER = None
_NODE_LABEL_ORDER_INDEX = {}
_EDGE_LABEL_ORDER_INDEX = {}




def _normalize_color(value):
    if isinstance(value, str) and value.startswith("#"):
        hex_value = value[1:]
        if len(hex_value) == 8:
            r = int(hex_value[0:2], 16)
            g = int(hex_value[2:4], 16)
            b = int(hex_value[4:6], 16)
            a = int(hex_value[6:8], 16) / 255.0
            return f"rgba({r},{g},{b},{a:.3f})"
        if len(hex_value) == 4:
            r = int(hex_value[0] * 2, 16)
            g = int(hex_value[1] * 2, 16)
            b = int(hex_value[2] * 2, 16)
            a = int(hex_value[3] * 2, 16) / 255.0
            return f"rgba({r},{g},{b},{a:.3f})"
        if len(hex_value) == 3:
            r = int(hex_value[0] * 2, 16)
            g = int(hex_value[1] * 2, 16)
            b = int(hex_value[2] * 2, 16)
            return f"#{r:02x}{g:02x}{b:02x}"
    return value

def _auto_color_for_key(key):
    digest = hashlib.md5(str(key).encode('utf-8')).hexdigest()
    r = int(digest[0:2], 16)
    g = int(digest[2:4], 16)
    b = int(digest[4:6], 16)
    return f"#{r:02x}{g:02x}{b:02x}"

def _color_for_key(key, *, palette, color_map, ordered_labels=None, ordered_index=None):
    if key in color_map:
        return _normalize_color(color_map[key])
    if ordered_labels is not None:
        ordered_index = ordered_index or {}
        if key in ordered_index:
            idx = ordered_index[key]
            if idx < len(palette):
                color = palette[idx]
            else:
                color = _auto_color_for_key(key)
            color = _normalize_color(color)
            color_map[key] = color
            return color
        reserved = min(len(ordered_labels), len(palette))
        used_colors = set(color_map.values())
        color = None
        for idx in range(reserved, len(palette)):
            candidate = _normalize_color(palette[idx])
            if candidate not in used_colors:
                color = candidate
                break
        if color is None:
            color = _normalize_color(_auto_color_for_key(key))
        color_map[key] = color
        return color
    if len(color_map) < len(palette):
        color = palette[len(color_map)]
    else:
        color = _auto_color_for_key(key)
    color = _normalize_color(color)
    color_map[key] = color
    return color

def _node_color(label):
    return _color_for_key(
        label,
        palette=_NODE_PALETTE,
        color_map=_NODE_COLOR_MAP,
        ordered_labels=_NODE_LABEL_ORDER,
        ordered_index=_NODE_LABEL_ORDER_INDEX,
    )

def _edge_color(label):
    return _color_for_key(
        label,
        palette=_EDGE_PALETTE,
        color_map=_EDGE_COLOR_MAP,
        ordered_labels=_EDGE_LABEL_ORDER,
        ordered_index=_EDGE_LABEL_ORDER_INDEX,
    )

def _node_label(graph, node):
    return graph.nodes[node].get('label', node)

def _edge_label(graph, u, v):
    data = graph.edges[u, v]
    label = data.get('label')
    if label is None:
        label = data.get('bond', data.get('type', data.get('weight', '')))
    return str(label)

def _marker_styles(graph, node_ids, selected, cut_nodes, cut_counts):
    selected = set(selected)
    cut_nodes = set(cut_nodes)
    fill_colors = [_node_color(_node_label(graph, n)) for n in node_ids]
    line_colors = ["black" if n in cut_nodes else "white" for n in node_ids]
    line_widths = [2 if cut_counts.get(n, 0) > 1 else 1 for n in node_ids]
    return fill_colors, line_colors, line_widths

def _axis_limits(pos, pad_ratio=0.1):
    if not pos:
        return (-1.0, 1.0), (-1.0, 1.0)
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    dx = max(max_x - min_x, 1e-6)
    dy = max(max_y - min_y, 1e-6)
    pad_x = dx * pad_ratio
    pad_y = dy * pad_ratio
    return (min_x - pad_x, max_x + pad_x), (min_y - pad_y, max_y + pad_y)


def _layout_components(graph, *, pad_ratio=0.3):
    components = list(nx.connected_components(graph))
    if not components:
        return {}
    if len(components) == 1:
        return nx.kamada_kawai_layout(graph)
    components.sort(key=lambda c: (-len(c), ",".join(sorted(str(n) for n in c))))
    comp_data = []
    max_dim = 0.0
    for nodes in components:
        subgraph = graph.subgraph(nodes)
        pos = nx.kamada_kawai_layout(subgraph)
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        min_x, max_x = min(xs), max(xs)
        min_y, max_y = min(ys), max(ys)
        cx = (min_x + max_x) / 2.0
        cy = (min_y + max_y) / 2.0
        centered = {n: (p[0] - cx, p[1] - cy) for n, p in pos.items()}
        width = max(max_x - min_x, 1e-6)
        height = max(max_y - min_y, 1e-6)
        max_dim = max(max_dim, width, height)
        comp_data.append((centered, width, height))
    pad = max(0.5, pad_ratio * max_dim)
    cols = max(1, int(math.ceil(math.sqrt(len(comp_data)))))
    positions = {}
    row_x = 0.0
    row_y = 0.0
    row_height = 0.0
    col = 0
    for centered, width, height in comp_data:
        if col >= cols:
            row_y -= row_height + pad
            row_x = 0.0
            row_height = 0.0
            col = 0
        for n, (x, y) in centered.items():
            positions[n] = (x + row_x, y + row_y)
        row_x += width + pad
        row_height = max(row_height, height)
        col += 1
    return positions


def _iter_cut_index_items(cut_index):
    if not cut_index:
        return
    for key, value in cut_index.items():
        if isinstance(value, dict):
            yield from _iter_cut_index_items(value)
        else:
            yield key, value


def _nodes_in_any_cut(graph, cut_index, cut_radius, cut_radii=None, max_cut_size=None, max_checks=2000):
    if not cut_index:
        return set(), {}, set()
    nodes = list(graph.nodes())
    sizes = sorted({len(k) for k, _ in _iter_cut_index_items(cut_index) if len(k) > 0})
    if max_cut_size is not None:
        sizes = [s for s in sizes if s <= max_cut_size]
    if cut_radii:
        cut_radii = sorted(set(int(r) for r in cut_radii))
    out = set()
    cut_counts = {}
    cut_pairs = set()
    checks = 0

    def _match(cut_nodes):
        if cut_radii:
            node = cut_index
            for radius in cut_radii:
                key = virtual_cut_signature(graph, cut_nodes, cut_radius=radius)
                node = node.get(key) if isinstance(node, dict) else None
                if not node:
                    return None
            return node
        key = virtual_cut_signature(graph, cut_nodes, cut_radius=cut_radius)
        return cut_index.get(key)

    for size in sizes:
        if size > len(nodes):
            continue
        for combo in itertools.combinations(nodes, size):
            if max_checks is not None and checks >= max_checks:
                return out, cut_counts, cut_pairs
            checks += 1
            if _match(combo):
                out.update(combo)
                for node in combo:
                    cut_counts[node] = cut_counts.get(node, 0) + 1
                if size > 1:
                    cut_pairs.update(itertools.combinations(combo, 2))
    return out, cut_counts, cut_pairs

def _graph_to_plotly_figure(
    graph,
    *,
    selected=None,
    title=None,
    seed=0,
    interactive=False,
    side_size=800,
    cut_nodes=None,
    cut_counts=None,
    cut_pairs=None,
):
    selected = set(selected or [])
    cut_nodes = set(cut_nodes or [])
    cut_counts = cut_counts or {}
    cut_pairs = set(cut_pairs or [])
    pos = _layout_components(graph)
    x_range, y_range = _axis_limits(pos)
    node_ids = list(graph.nodes())
    node_labels = [_node_label(graph, n) for n in node_ids]
    for lbl in sorted(set(node_labels), key=lambda x: str(x)):
        _node_color(lbl)
    text_colors = ["black" if n in cut_nodes else "white" for n in node_ids]
    fill_colors, line_colors, line_widths = _marker_styles(graph, node_ids, selected, cut_nodes, cut_counts)
    click_enabled = False
    try:
        fig = go.FigureWidget() if interactive else go.Figure()
        click_enabled = bool(interactive)
    except Exception:
        fig = go.Figure()
        click_enabled = False

    if cut_pairs:
        cut_x = []
        cut_y = []
        for u, v in cut_pairs:
            if u not in pos or v not in pos:
                continue
            cut_x.extend([pos[u][0], pos[v][0], None])
            cut_y.extend([pos[u][1], pos[v][1], None])
        fig.add_trace(
            go.Scatter(
                x=cut_x,
                y=cut_y,
                mode="lines",
                line=dict(width=1, color="rgba(140,140,140,0.4)", dash="dot"),
                hoverinfo="none",
                name="virtual_cut",
            )
        )

    # Edge traces grouped by edge label for consistent coloring.
    edge_groups = {}
    for u, v in graph.edges():
        lbl = _edge_label(graph, u, v)
        edge_groups.setdefault(lbl, []).append((u, v))
    for lbl in sorted(edge_groups, key=str):
        _edge_color(lbl)
    for lbl, edges in edge_groups.items():
        edge_x = []
        edge_y = []
        for u, v in edges:
            edge_x.extend([pos[u][0], pos[v][0], None])
            edge_y.extend([pos[u][1], pos[v][1], None])
        fig.add_trace(
            go.Scatter(
                x=edge_x,
                y=edge_y,
                mode="lines",
                line=dict(width=8, color=_edge_color(lbl)),
                hoverinfo="none",
                name=f"edge:{lbl}",
            )
        )

    node_x = [pos[n][0] for n in node_ids]
    node_y = [pos[n][1] for n in node_ids]
    node_positions = {n: (pos[n][0], pos[n][1]) for n in node_ids}
    node_trace_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=node_x,
            y=node_y,
            mode="markers+text",
            text=[str(lbl) for lbl in node_labels],
            textposition="middle center",
            hoverinfo="text",
            hovertext=[f"cuts: {cut_counts.get(n, 0)}" for n in node_ids],
            marker=dict(size=int(18 * 1.1), color=fill_colors, line=dict(width=line_widths, color=line_colors)),
            textfont=dict(color=text_colors),
            name="nodes",
        )
    )
    selected_x = [pos[n][0] for n in node_ids if n in selected]
    selected_y = [pos[n][1] for n in node_ids if n in selected]
    selected_trace_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(
            x=selected_x,
            y=selected_y,
            mode="markers",
            marker=dict(size=24, color="rgba(0,0,0,0)", line=dict(width=4, color="red")),
            hoverinfo="none",
            name="selected_outline",
        )
    )
    fig.update_layout(
        showlegend=False,
        margin=dict(l=10, r=10, t=30, b=10),
        title=title,
        title_x=0.5,
        width=int(side_size),
        height=int(side_size),
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis=dict(visible=False, scaleanchor="y", scaleratio=1, range=x_range),
        yaxis=dict(visible=False, range=y_range),
    )
    return fig, node_ids, click_enabled, node_trace_idx, selected_trace_idx, node_positions

def interactive_virtual_cut_selector(
    seed,
    generator,
    feasibility_estimator,
    *,
    n_virtual_candidates: int = 30,
    max_virtual_cut_evaluations: int = 200,
    max_attempts: int = 5,
    columns: int = 5,
    layout_seed: int = 0,
    side_size: int = 800,
    candidate_side_size: int = 1200,
    domain_draw=draw_molecules,
    allow_superset: bool = True,
    max_superset_checks: int = 200,
    max_cut_size: int = 3,
    node_labels=None,
    edge_labels=None,
    highlight_max_checks: int = 2000,
):
    """Interactively choose cut nodes, generate candidates, and iterate.

    If node_labels or edge_labels are provided, palette colors are assigned
    in the given order; unlisted labels use the next available colors.
    """
    cut_radius = getattr(generator, "cut_radius", None)
    cut_radii = getattr(generator, "cut_radii", None)
    cut_index = getattr(generator, "cut_index", {})
    global _NODE_LABEL_ORDER, _EDGE_LABEL_ORDER
    global _NODE_LABEL_ORDER_INDEX, _EDGE_LABEL_ORDER_INDEX
    if node_labels is None:
        _NODE_LABEL_ORDER = None
        _NODE_LABEL_ORDER_INDEX = {}
    else:
        _NODE_LABEL_ORDER = list(node_labels)
        _NODE_LABEL_ORDER_INDEX = {label: idx for idx, label in enumerate(_NODE_LABEL_ORDER)}
        _NODE_COLOR_MAP.clear()
    if edge_labels is None:
        _EDGE_LABEL_ORDER = None
        _EDGE_LABEL_ORDER_INDEX = {}
    else:
        _EDGE_LABEL_ORDER = list(edge_labels)
        _EDGE_LABEL_ORDER_INDEX = {label: idx for idx, label in enumerate(_EDGE_LABEL_ORDER)}
        _EDGE_COLOR_MAP.clear()
    state = {
        "seed": seed,
        "selected": set(),
        "candidates": [],
        "node_ids": [],
        "node_positions": {},
        "node_trace": None,
        "selected_trace": None,
        "click_enabled": False,
    }
    main_out = widgets.Output()
    mol_out = widgets.Output()
    mol_out_container = widgets.HBox(
        [mol_out],
        layout=widgets.Layout(
            width=f"{side_size}px",
            justify_content="center",
            align_items="center",
        ),
    )
    selected_out = widgets.Output()
    candidates_out = widgets.Output()
    status = widgets.HTML(value="")

    status_timer = {"timer": None}

    def set_status(message, *, timeout=None):
        if status.value == message:
            status.value = ""
        status.value = message
        timer = status_timer.get("timer")
        if timer is not None:
            timer.cancel()
            status_timer["timer"] = None
        if timeout is not None:
            def clear():
                status.value = ""
            timer = threading.Timer(timeout, clear)
            timer.daemon = True
            timer.start()
            status_timer["timer"] = timer

    def render_seed():
        cut_nodes, cut_counts, cut_pairs = _nodes_in_any_cut(state["seed"], cut_index, cut_radius, cut_radii=cut_radii, max_cut_size=max_cut_size, max_checks=highlight_max_checks)
        fig, node_ids, click_enabled, node_trace_idx, selected_trace_idx, node_positions = _graph_to_plotly_figure(
            state["seed"],
            selected=state["selected"],
            title="",
            seed=layout_seed,
            interactive=True,
            side_size=side_size,
            cut_nodes=cut_nodes,
            cut_counts=cut_counts,
            cut_pairs=cut_pairs,
        )
        state["node_ids"] = node_ids
        state["node_positions"] = node_positions
        state["click_enabled"] = click_enabled
        state["node_trace"] = fig.data[node_trace_idx] if node_trace_idx is not None else None
        state["selected_trace"] = fig.data[selected_trace_idx] if selected_trace_idx is not None else None
        current = [n for n in state["selected"] if n in node_ids]
        state["selected"] = set(current)
        with main_out:
            clear_output(wait=True)
            display(fig)
        with mol_out:
            clear_output(wait=True)
            try:
                domain_draw([state["seed"]], n_graphs_per_line=1, size=21)
            except Exception as exc:
                print(f"mol_display failed: {exc}")
        with selected_out:
            clear_output(wait=True)
            print(f"Selected nodes: {sorted(state['selected'])}")
        if click_enabled and state["node_trace"] is not None:
            node_trace = state["node_trace"]
            def handle_click(trace, points, selector):
                for idx in points.point_inds:
                    node_id = node_ids[idx]
                    if node_id in state["selected"]:
                        state["selected"].remove(node_id)
                    else:
                        state["selected"].add(node_id)
                fill_colors, line_colors, line_widths = _marker_styles(
                    state["seed"],
                    node_ids,
                    state["selected"],
                    cut_nodes,
                    cut_counts,
                )
                node_trace.marker.color = fill_colors
                node_trace.marker.line.color = line_colors
                node_trace.marker.line.width = line_widths
                if state.get("selected_trace") is not None:
                    state["selected_trace"].x = [state["node_positions"][n][0] for n in node_ids if n in state["selected"]]
                    state["selected_trace"].y = [state["node_positions"][n][1] for n in node_ids if n in state["selected"]]
                with selected_out:
                    clear_output(wait=True)
                    print(f"Selected nodes: {sorted(state['selected'])}")
            node_trace.on_click(handle_click)
        if click_enabled:
            set_status("<b>Click nodes or use the selector.</b>")
        else:
            set_status("<b>Click disabled (FigureWidget unavailable); use the selector.</b>")

    def choose_candidate(idx, *, clear_candidates=True, update_status=True):
        state["seed"] = state["candidates"][idx]
        state["selected"] = set()
        if update_status:
            set_status(f"Selected candidate {idx}.")
        render_seed()
        if clear_candidates:
            with candidates_out:
                clear_output(wait=True)

    def show_candidates():
        with candidates_out:
            clear_output(wait=True)
            if not state["candidates"]:
                return
            rows = []
            grid_cols = max(1, int(columns))
            candidate_side_local = max(240, int(candidate_side_size / max(1, grid_cols)))
            for i in range(0, len(state["candidates"]), max(1, grid_cols)):
                row_widgets = []
                for j, graph in enumerate(state["candidates"][i : i + grid_cols], start=i):
                    fig, _node_ids, _click, _node_trace_idx, _selected_trace_idx, _node_positions = _graph_to_plotly_figure(
                        graph,
                        selected=None,
                        title=f"Candidate {j}",
                        seed=layout_seed,
                        interactive=False,
                        side_size=candidate_side_local,
                        cut_counts=None,
                    )
                    fig_out = widgets.Output()
                    with fig_out:
                        clear_output(wait=True)
                        display(fig)
                    btn = widgets.Button(description=f"Use {j}", layout=widgets.Layout(margin="0 auto"))
                    btn.on_click(lambda _btn, idx=j: choose_candidate(idx))
                    row_widgets.append(
                        widgets.VBox(
                            [fig_out, btn],
                            layout=widgets.Layout(
                                align_items="center",
                                border="0.5px solid #9a9a9a",
                                padding="6px",
                                margin="4px",
                            ),
                        )
                    )
                rows.append(widgets.HBox(row_widgets))
            display(widgets.VBox(rows))

    def generate_candidates(_btn):
        cut_nodes = list(state["selected"])
        if max_cut_size is not None and len(cut_nodes) > max_cut_size:
            set_status(f"<b>Cut size {len(cut_nodes)} exceeds max_cut_size={max_cut_size}.</b>")
            return
        if not cut_nodes:
            set_status("<b>Select at least one node for the cut.</b>")
            return
        set_status("<b>&#x23F3; Generating candidates...</b>")
        candidates = []
        for _attempt in range(max(1, int(max_attempts))):
            candidates = generator.propose_virtual_candidates_at_cut_nodes(
                state["seed"],
                cut_nodes,
                n_samples=n_virtual_candidates,
                allow_superset=allow_superset,
                max_superset_checks=max_superset_checks,
            )
            candidates = feasibility_estimator.filter(candidates)
            candidates = GraphHashDeduper().fit_filter(candidates)
            if candidates:
                break
        if not candidates:
            state["selected"] = set()
            render_seed()
            set_status("<b>No virtual candidates found for this cut.</b>", timeout=2)
            return
        state["candidates"] = candidates
        set_status(f"Generated {len(candidates)} candidates.")
        show_candidates()
        choose_candidate(0, clear_candidates=False, update_status=False)

    def generate_candidates_auto(_btn):
        set_status("<b>&#x23F3; Generating candidates (auto)...</b>")
        candidates = []
        for _attempt in range(max(1, int(max_attempts))):
            candidates = generator.propose_virtual_candidates(
                state["seed"],
                n_virtual_candidates=n_virtual_candidates,
                max_virtual_cut_evaluations=max_virtual_cut_evaluations,
                max_cut_size=max_cut_size,
            )
            candidates = feasibility_estimator.filter(candidates)
            candidates = GraphHashDeduper().fit_filter(candidates)
            if candidates:
                break
        if not candidates:
            set_status("<b>No virtual candidates found for this graph.</b>", timeout=2)
            return
        state["candidates"] = candidates
        set_status(f"Generated {len(candidates)} candidates.")
        show_candidates()
        choose_candidate(0, clear_candidates=False, update_status=False)

    def clear_selection(_btn):
        state["selected"] = set()
        set_status("Cleared selection.")
        render_seed()

    def delete_selected(_btn):
        if not state["selected"]:
            set_status("Select nodes to delete.")
            return
        state["seed"].remove_nodes_from(list(state["selected"]))
        state["selected"] = set()
        state["candidates"] = []
        with candidates_out:
            clear_output(wait=True)
        set_status("Deleted selected nodes.")
        render_seed()

    generate_btn = widgets.Button(description="Generate candidates", button_style="success")
    auto_btn = widgets.Button(description="Generate (auto)", button_style="info")
    clear_btn = widgets.Button(description="Clear selection")
    delete_btn = widgets.Button(description="Delete selected", button_style="danger")
    generate_btn.on_click(generate_candidates)
    auto_btn.on_click(generate_candidates_auto)
    clear_btn.on_click(clear_selection)
    delete_btn.on_click(delete_selected)

    render_seed()
    ui = widgets.VBox([
        widgets.HBox([generate_btn, auto_btn, clear_btn, delete_btn]),
        status,
        selected_out,
        widgets.HBox([main_out, mol_out_container], layout=widgets.Layout(align_items="center")),
        candidates_out,
    ])
    display(ui)
    return ui


In [ ]:
from coco_grape.data_loader.mol.mol_loader import PubChemLoader
from coco_grape.data_loader.loader import SupervisedDataSetLoader

assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[1]
#assay_id = '488975' #active 2,634
size = 300
print(f"size: {size}")
use_equalized = True

def pubchem_loader():
    return PubChemLoader().load(assay_id)

graphs, targets = SupervisedDataSetLoader(
    pubchem_loader,
    size=size,
    use_equalized=use_equalized,
).load()
targets = np.array(targets)

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)


In [ ]:
node_label_set = set([graph.nodes[u]['label'] for graph in graphs for u in graph.nodes()])
edge_label_set = set([graph.edges[u,v]['label'] for graph in graphs for u,v in graph.edges()])
print(node_label_set)
print(edge_label_set)

In [ ]:
from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator
df = compose(neighborhood(radius=2), unlabel())
fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
df = add(neighborhood(radius=1), cycle())
fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
feasibility_estimators = [fe1, fe2]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)

nbits=14
#decomposition_function = add(cycle(), neighborhood(radius=(0,2)))
#decomposition_function = add(node(), edge(), cycle())
#decomposition_function = add(node(), compose_product(binary_combination(distance=3), neighborhood(), node()))
decomposition_function = add(node(), edge(), cycle(), neighborhood(radius=(1,2)))
from abstractgraph_generative.autoregressive import AutoregressiveGraphGenerator
generator = AutoregressiveGraphGenerator(
    feasibility_estimator, 
    nbits, 
    decomposition_function, 
    cut_radius=2, 
    n_virtual_candidates=30, 
    use_similarity_selection=False,
    n_pruning_iterations=20, 
    verbose=True,
    n_jobs=-1)
generator.store(graphs)

In [ ]:
generator_graphs = generator.get_generators(size=1)
from coco_grape.visualizer.mol_display import draw_molecules as display_graphs
_ = display_graphs(generator_graphs)
generator.fit(generator_graphs)

In [ ]:
seed = generator.donors[random.randrange(len(generator.donors))]
_ = display_graphs([seed])

In [ ]:
from coco_grape.visualizer.mol_display import draw_molecules
node_labels = ['C','S','N','O','Br']
edge_labels = ['1', 'AROMATIC', '2','3']
ui = interactive_virtual_cut_selector(
    seed=seed,
    generator=generator,
    feasibility_estimator=feasibility_estimator,
    node_labels=node_labels,
    edge_labels=edge_labels,
    n_virtual_candidates=30,
    max_attempts=10,
    side_size=700,
    candidate_side_size=1500,
    columns=3,
    domain_draw=draw_molecules,
)